# Детекция курящих людей — YOLO11s (person / cigarette / vape / smoke)

Минимальная версия под первый вариант обучения: только необходимые шаги, без скачивания/слияния сторонних датасетов.

**Перед запуском:** `Runtime` → `Change runtime type` → `GPU` (T4).

## 1. Установка библиотек

In [ ]:
!pip install -q ultralytics


## 2. Подключение Google Drive
При запуске появится запрос на авторизацию — разреши доступ к аккаунту, куда заливал `my_dataset`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/smoking_detection'
import os
os.makedirs(WORK_DIR, exist_ok=True)


## 3. Диагностика датасета
Проверка перед обучением: сколько картинок и файлов разметки в train/val. Если `labels` пустая — обучение упадёт с ошибкой `No labels found`.

In [ ]:
DATASET_DIR = f'{WORK_DIR}/my_dataset'

for split in ['train', 'val']:
    images_dir = f'{DATASET_DIR}/images/{split}'
    labels_dir = f'{DATASET_DIR}/labels/{split}'
    images = os.listdir(images_dir) if os.path.exists(images_dir) else []
    labels = os.listdir(labels_dir) if os.path.exists(labels_dir) else []
    print(f'--- {split} ---')
    print('Картинок:', len(images), '| Файлов разметки:', len(labels))
    if not labels:
        print('⚠️ Разметка отсутствует для этого split!')
    print()


## 4. Путь к data.yaml
Файл должен лежать в корне `my_dataset`, рядом с `images/` и `labels/`.

In [ ]:
DATA_YAML = f'{DATASET_DIR}/data.yaml'
with open(DATA_YAML) as f:
    print(f.read())


## 5. Обучение YOLO11s
`imgsz=1280` — критично для мелких объектов (сигарета/дым на дистанции). `patience=30` — ранняя остановка при отсутствии улучшений.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data=DATA_YAML,
    epochs=150,
    imgsz=1280,
    batch=16,
    device=0,
    patience=30,
    project=f'{WORK_DIR}/runs',
    name='smoking_yolo11s'
)


## 5b. Если обучение прервалось на середине — продолжить с чекпоинта
Запускай вместо ячейки 5, только если уже есть `last.pt` от незавершённой попытки.

In [ ]:
# from ultralytics import YOLO
# model = YOLO(f'{WORK_DIR}/runs/smoking_yolo11s/weights/last.pt')
# results = model.train(resume=True)


## 6. Валидация

In [ ]:
metrics = model.val()
print('mAP50-95:', metrics.box.map)
print('mAP50:', metrics.box.map50)
print('mAP50-95 по классам:', metrics.box.maps)


## 7. Предсказания на тестовых фото без разметки
Прогоняем модель на фото и смотрим, что она находит — без расчёта метрик.

In [ ]:
TEST_IMAGES_DIR = f'{DATASET_DIR}/images/test'  # поправь путь, если папка называется иначе

results_pred = model.predict(
    source=TEST_IMAGES_DIR,
    conf=0.25,
    imgsz=1280,
    save=True,
    project=f'{WORK_DIR}/runs',
    name='predict_test'
)
print('Готово. Результаты сохранены в:', f'{WORK_DIR}/runs/predict_test')


## 8. Посмотреть результаты прямо в Colab

In [ ]:
from IPython.display import Image, display
import glob

for img_path in glob.glob(f'{WORK_DIR}/runs/predict_test/*.jpg')[:10]:
    display(Image(filename=img_path))
